In [0]:
&ip install dbt-databricks



In [0]:
import os
import subprocess

# Crea el directorio ~/.dbt si no existe antes de ejecutar dbt
profiles_dir = os.path.expanduser("~/.dbt")
os.makedirs(profiles_dir, exist_ok=True)

# Crea profiles.yml con el perfil 'databricks_profile'
#databricks_token = os.environ.get("DATABRICKS_TOKEN", "")
#databricks_host  = os.environ.get("DATABRICKS_HOST",  "")
#databricks_http  = os.environ.get("DATABRICKS_HTTP",  "")
#databricks_catalog = os.environ.get("DATABRICKS_CATALOG", "")

profiles_yml = f"""
databricks_profile:
  target: dev
  outputs:
    dev:
      type: databricks
      host: {databricks_host}
      http_path: {databricks_http}
      token: {databricks_token}
      schema: default
      catalog: {databricks_catalog}
      threads: 1
""".strip()

with open(f"{profiles_dir}/profiles.yml", "w") as f:
    f.write(profiles_yml)

# Crea un modelo dbt usando Jinja para mostrar el potencial de plantillas
modelo_template_sql = """
{% set valores = [1, 2, 3, 4] %}
SELECT
    {{ valores | join(', ') }} AS numeros,
    COUNT(*) AS cantidad
FROM (
    SELECT {{ valores | join(' UNION ALL SELECT ') }} AS numero
)
"""

# Escribe el modelo en el directorio de modelos de dbt
os.makedirs("mi_proyecto_dbt/models", exist_ok=True)
with open("mi_proyecto_dbt/models/ejemplo_template.sql", "w") as f:
    f.write(modelo_template_sql)

# Crea dbt_project.yml con el perfil correcto
dbt_project_yml = """
name: mi_proyecto_dbt
version: '1.0.0'
config-version: 2

profile: databricks_profile

model-paths: ["models"]

models:
  mi_proyecto_dbt:
    +materialized: view
""".strip()

with open("mi_proyecto_dbt/dbt_project.yml", "w") as f:
    f.write(dbt_project_yml)

# Ejecuta el modelo dbt con plantilla y muestra el error si ocurre
result = subprocess.run(
    ["dbt", "run", "--project-dir", "mi_proyecto_dbt", "--profiles-dir", profiles_dir, "--select", "ejemplo_template"],
    capture_output=True,
    text=True
)
result

In [0]:
import subprocess
import os

PROJECT_DIR = "mi_proyecto_dbt"
profiles_dir = os.path.expanduser("~/.dbt")

# Compila el modelo dbt usando la configuración y planilla creada
compile_result = subprocess.run(
    [
        "dbt", "compile",
        "--project-dir", PROJECT_DIR,
        "--profiles-dir", profiles_dir,
        "--select", "ejemplo_template"
    ],
    capture_output=True,
    text=True
)

# Muestra el SQL compilado generado por la plantilla Jinja
compiled_path = f"{PROJECT_DIR}/target/compiled/mi_proyecto_dbt/models/ejemplo_template.sql"
if os.path.exists(compiled_path):
    with open(compiled_path) as f:
        print(f.read())
else:
    print(compile_result.stdout)

In [0]:
import os

# Crea un modelo dbt usando Jinja y la tabla samples.nyctaxi.trips
modelo_template_sql = """
{% set zips = [10001, 10002, 10003] %}
SELECT
    pickup_zip,
    COUNT(*) AS total_viajes,
    AVG(trip_distance) AS distancia_promedio
FROM samples.nyctaxi.trips
WHERE pickup_zip IN ({{ zips | join(', ') }})
GROUP BY pickup_zip
ORDER BY pickup_zip
"""

os.makedirs("mi_proyecto_dbt/models", exist_ok=True)
with open("mi_proyecto_dbt/models/ejemplo_trips_template.sql", "w") as f:
    f.write(modelo_template_sql)

# Compila el modelo dbt usando la configuración y plantilla creada
import subprocess

PROJECT_DIR = "mi_proyecto_dbt"
profiles_dir = os.path.expanduser("~/.dbt")

compile_result = subprocess.run(
    [
        "dbt", "compile",
        "--project-dir", PROJECT_DIR,
        "--profiles-dir", profiles_dir,
        "--select", "ejemplo_trips_template"
    ],
    capture_output=True,
    text=True
)

# Muestra el SQL compilado generado por la plantilla Jinja
compiled_path = f"{PROJECT_DIR}/target/compiled/mi_proyecto_dbt/models/ejemplo_trips_template.sql"
if os.path.exists(compiled_path):
    with open(compiled_path) as f:
        print(f.read())
else:
    print(compile_result.stdout)

In [0]:
import os

# Crea un modelo dbt usando Jinja para reutilizar el template con diferentes códigos postales
modelo_template_sql_reuso = """
{% set zips = [10004, 10005, 10006] %}
SELECT
    pickup_zip,
    COUNT(*) AS total_viajes,
    AVG(trip_distance) AS distancia_promedio
FROM samples.nyctaxi.trips
WHERE pickup_zip IN ({{ zips | join(', ') }})
GROUP BY pickup_zip
ORDER BY pickup_zip
"""

os.makedirs("mi_proyecto_dbt/models", exist_ok=True)
with open("mi_proyecto_dbt/models/ejemplo_trips_template_reuso.sql", "w") as f:
    f.write(modelo_template_sql_reuso)

# Compila el modelo dbt usando la configuración y plantilla creada
PROJECT_DIR = "mi_proyecto_dbt"
profiles_dir = os.path.expanduser("~/.dbt")

compile_result = subprocess.run(
    [
        "dbt", "compile",
        "--project-dir", PROJECT_DIR,
        "--profiles-dir", profiles_dir,
        "--select", "ejemplo_trips_template_reuso"
    ],
    capture_output=True,
    text=True
)

# Muestra el SQL compilado generado por la plantilla Jinja reutilizada
compiled_path = f"{PROJECT_DIR}/target/compiled/mi_proyecto_dbt/models/ejemplo_trips_template_reuso.sql"
if os.path.exists(compiled_path):
    with open(compiled_path) as f:
        print(f.read())
else:
    print(compile_result.stdout)